# Retail Sales Analysis with Python

**Goal:** turn 10,000 transactional records into validated, analysis-ready data and decision-oriented findings using pandas and Matplotlib.

> Currency is not documented in the source, so monetary values are reported as **source units** rather than assigning a currency.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / 'data/raw/sales_orders.csv', parse_dates=['order_date','ship_date'])
df.head()

## 1. Data quality

In [ ]:
quality = {
    'rows': len(df),
    'duplicate_rows': df.duplicated().sum(),
    'missing_cells': df.isna().sum().sum(),
    'ship_before_order': (df['ship_date'] < df['order_date']).sum(),
    'negative_profit_orders': (df['profit'] < 0).sum(),
}
quality

## 2. Feature engineering

In [ ]:
df = df.drop_duplicates().copy()
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.to_period('M').astype(str)
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days
df['profit_status'] = np.where(df['profit'] < 0, 'Loss', 'Profit')
df.head()

## 3. KPI summary

In [ ]:
total_sales = df['sales'].sum()
total_profit = df['profit'].sum()
kpis = {
    'Total Sales': total_sales,
    'Total Profit': total_profit,
    'Profit Margin %': total_profit / total_sales * 100,
    'Orders': len(df),
    'Units Sold': df['quantity'].sum(),
    'Average Order Value': total_sales / len(df),
}
pd.Series(kpis).round(2)

## 4. Annual performance

In [ ]:
yearly = df.groupby('order_year').agg(sales=('sales','sum'), profit=('profit','sum'))
yearly['profit_margin_pct'] = yearly['profit'] / yearly['sales'] * 100
yearly['yoy_sales_growth_pct'] = yearly['sales'].pct_change() * 100
yearly.round(2)

In [ ]:
yearly['sales'].plot(marker='o', title='Annual Sales Trend', ylabel='Sales (source units)')
plt.grid(axis='y', alpha=.25)
plt.show()

## 5. Category and regional performance

In [ ]:
category = df.groupby('category').agg(sales=('sales','sum'), profit=('profit','sum'))
category['profit_margin_pct'] = category['profit'] / category['sales'] * 100
category.sort_values('sales', ascending=False).round(2)

In [ ]:
region = df.groupby('region').agg(sales=('sales','sum'), profit=('profit','sum'))
region['profit_margin_pct'] = region['profit'] / region['sales'] * 100
region.sort_values('sales', ascending=False).round(2)

## 6. Discount vs profitability

In [ ]:
discount = df.groupby('discount').agg(sales=('sales','sum'), profit=('profit','sum'), orders=('order_id','count'))
discount['profit_margin_pct'] = discount['profit'] / discount['sales'] * 100
discount.round(2)

In [ ]:
discount['profit_margin_pct'].plot(marker='o', title='Discount Level vs Profit Margin', xlabel='Discount', ylabel='Profit Margin (%)')
plt.grid(alpha=.25)
plt.show()

## 7. Key findings

- Total sales are **10.03M source units** with **1.71M** profit and a **17.05%** aggregate margin.
- 2022 sales grew **3.36%** versus 2021 after a decline in 2021.
- **Furniture** generated the highest category sales, while **Technology** had the highest category margin.
- **East** generated the highest regional sales, while **West** had the highest regional margin.
- Aggregate margin declines from **20.35% at 0% discount** to **13.99% at 30% discount**. This is an association in this dataset, not proof that discounting alone caused the margin difference.
- The dataset uses unique customer and product IDs for every row, so repeat-customer and SKU-level retention analysis is intentionally not claimed.